# Colab runner: sparse-view 3DGS (DNGaussian + dropout)

Run top to bottom at the start of every session. Runtime > Change runtime type > **T4 GPU**.

| Cell | When | What |
| --- | --- | --- |
| 1-2 | Every session | Check GPU, mount Drive (Disha's first run also creates the folder tree) |
| 3 | Every session | Get the code: team repo at a tag, or upstream DNGaussian until the team repo exists |
| 3b | Every session | Apply build fixes for newer compilers (no-op if already applied) |
| 4-5 | Every session | Install CUDA extensions (built once, then cached as wheels on Drive) and Python packages |
| D | **Once, ever** (Disha, Mon 12 Oct) | Download LLFF, generate DPT depth, upload `llff_with_depth.tar` |
| 6 | Every session | Extract the data archive to the fast local disk |
| S | Mon 12 Oct | Smoke test / timing run on `fern`: gives the minutes-per-run number *m* |
| 7a | Tue 13 Oct (Disha) | Reproduce DNGaussian on all 8 scenes at tag `upstream-baseline` |
| 7b | From Wed 14 Oct 1 PM | Run your shard of the main matrix at tag `v1.0-exp` |

Secrets (key icon on the left): add `GH_TOKEN` (classic token, `repo` scope) and switch on notebook access. Never paste a token into a cell.

In [ ]:
# Cell 1: check the GPU and versions. nvcc and torch CUDA major versions must match.
!nvidia-smi
import torch, sys
print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu", torch.cuda.get_device_name(0))
!nvcc --version | tail -1

In [ ]:
# Cell 2: mount Drive. Disha (folder owner) sets CREATE_FOLDERS = True on her first run only.
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/sparse3dgs'
CREATE_FOLDERS = False
if CREATE_FOLDERS:
    for sub in ['data', 'weights', 'wheels', 'outputs/llff_3v', 'outputs/repro_upstream', 'logs']:
        os.makedirs(f'{ROOT}/{sub}', exist_ok=True)
    print('Created', ROOT, '- now share it with Aayush as Editor in Drive')
assert os.path.isdir(ROOT), "sparse3dgs not found: add the shared folder to My Drive as a shortcut first (Organize > Add shortcut)"
print(sorted(os.listdir(ROOT)))

In [ ]:
# Cell 3: get the code at a fixed tag.
#   upstream-baseline : Disha's reproduction (Tue)      v1.0-exp : main matrix (from Wed 1 PM)
#   USE_TEAM_REPO = False only until you have access to dngs-dropout: clones the public DNGaussian
#   plus its two submodules, which is enough to build the wheels and run the smoke test.
USE_TEAM_REPO = True
OWNER, TAG = 'aayxsh23', 'upstream-baseline'
CODE = '/content/dngs-dropout'
!rm -rf {CODE}
if USE_TEAM_REPO:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
    !git clone -q https://{tok}@github.com/{OWNER}/dngs-dropout.git {CODE}
    %cd {CODE}
    !git checkout -q {TAG} && git log -1 --oneline
else:
    !git clone -q --depth 1 https://github.com/Fictionarry/DNGaussian.git {CODE}
    %cd {CODE}
    !git clone -q --recursive https://github.com/ashawkey/diff-gaussian-rasterization.git submodules/diff-gaussian-rasterization
    !git clone -q https://gitlab.inria.fr/bkerbl/simple-knn.git submodules/simple-knn
    !git log -1 --oneline

In [ ]:
# Cell 3b: build fixes for newer compilers and PyTorch 2.x. Idempotent; tell Aayush to commit
# the same edits in the team repo so this cell becomes a no-op.
import re, pathlib
def patch(path, fn):
    p = pathlib.Path(path)
    if not p.exists(): print('missing', path); return
    s = p.read_text(); t = fn(s)
    if t != s: p.write_text(t); print('patched', path)
def add_include(inc):
    return lambda s: s if inc in s else inc + '\n' + s
patch('submodules/simple-knn/simple_knn.cu', add_include('#include <float.h>'))
patch('submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h', add_include('#include <cstdint>'))
# torch-ngp style encoders ask for C++14, PyTorch >= 2.1 needs C++17
for f in ['gridencoder/setup.py', 'gridencoder/backend.py', 'shencoder/setup.py', 'shencoder/backend.py']:
    patch(f, lambda s: s.replace('-std=c++14', '-std=c++17').replace('/std:c++14', '/std:c++17'))

In [ ]:
%%bash
# Cell 4: install the CUDA extensions, building them only once per PyTorch version.
# Training needs all four: rasterizer, simple-knn, gridencoder and shencoder (DNGaussian's neural renderer).
set -e
cd /content/dngs-dropout
TV=$(python -c "import torch;print(torch.__version__)")
W=/content/drive/MyDrive/sparse3dgs/wheels/torch-$TV
if ! ls $W/*.whl >/dev/null 2>&1; then
  echo "No cached wheels for torch $TV: building (first time takes a while)"
  mkdir -p $W
  export TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6"
  pip wheel --no-build-isolation --no-deps -w $W \
      ./submodules/diff-gaussian-rasterization ./submodules/simple-knn ./gridencoder ./shencoder
fi
pip install -q --force-reinstall --no-deps $W/*.whl
python -c "import diff_gaussian_rasterization, simple_knn._C, _gridencoder, _shencoder; print('extensions ok')"

In [ ]:
# Cell 5: Python packages. Add whatever an ImportError names, then commit the list as requirements_colab.txt.
# timm and lpips are left out on purpose: the repo ships lpipsPyTorch, and Cell D pins its own timm.
!pip install plyfile opencv-python-headless imageio pyyaml tensorboard scikit-image 2>&1 | tail -5

## Cell D: generate the depth data, ONCE (Disha, Monday)

Both of you must train on identical DPT depth maps, so this cell refuses to run if the archive already exists.
It downloads LLFF from the link in the DNGaussian README, runs the repo's own `dpt/get_depth_map_for_llff_dtu.py`
(DPT_Hybrid via torch.hub, written to `<scene>/depth_maps/`), checks one depth map per image, and uploads one archive.

If the download fails with a Google Drive quota error: open the link in a browser, download `nerf_llff_data`,
upload the zip to `sparse3dgs/data/`, and set `LOCAL_ZIP` to its path.

In [ ]:
# Cell D: one-time depth generation
import os, glob, subprocess, hashlib, json
TAR = f'{ROOT}/data/llff_with_depth.tar'
LOCAL_ZIP = None          # e.g. f'{ROOT}/data/nerf_llff_data.zip' if you uploaded it yourself
SCENES = ['fern','flower','fortress','horns','leaves','orchids','room','trex']
assert not os.path.exists(TAR), 'llff_with_depth.tar already exists: depth is generated exactly once. Delete it only if you mean to regenerate.'

# download + unzip, skipped if a previous (interrupted) run already prepared /content/build/llff
if not all(os.path.isdir(f'/content/build/llff/{s}/images') for s in SCENES):
    !pip install -q gdown
    os.makedirs('/content/raw', exist_ok=True)
    zip_path = LOCAL_ZIP or '/content/raw/llff.zip'
    if not LOCAL_ZIP:
        !gdown 11PhkBXZZNYTD2emdG1awALlhCnkq7aN- -O {zip_path}
    !cd /content/raw && unzip -q -o {zip_path}
    # find the folder holding the 8 scenes and move it to /content/build/llff
    src = [d for d in glob.glob('/content/raw/**/fern', recursive=True) if os.path.isdir(d + '/sparse')]
    assert src, 'fern/ with sparse/ not found in the download'
    llff_src = os.path.dirname(src[0])
    os.makedirs('/content/build', exist_ok=True)
    !rm -rf /content/build/llff && mv {llff_src} /content/build/llff
assert all(os.path.isdir(f'/content/build/llff/{s}/images') for s in SCENES), os.listdir('/content/build/llff')

# DPT_Hybrid from torch.hub. The MiDaS hub code was written for timm 0.6.x; newer timm can break the hybrid model.
!pip install -q "timm==0.6.13"
# torch.hub asks "Do you trust this repository?" for intel-isl/MiDaS and waits for an answer that never comes.
# Pre-trust it, and give the script no stdin so any other prompt fails loudly instead of hanging.
import torch
os.makedirs(torch.hub.get_dir(), exist_ok=True)
with open(os.path.join(torch.hub.get_dir(), 'trusted_list'), 'a') as f:
    f.write('intel-isl_MiDaS\n')
%cd /content/dngs-dropout/dpt
# A few minutes: DPT weights download first, then "k, img.shape" lines per image.
!python -u get_depth_map_for_llff_dtu.py --root_path /content/build/llff --benchmark LLFF < /dev/null 2>&1 | grep --line-buffered -v "image_paths\|output_path"
%cd /content/dngs-dropout

# check: one depth map per image, per scene
summary = {}
for s in SCENES:
    imgs = [p for p in glob.glob(f'/content/build/llff/{s}/images/*') if p.lower().endswith('.jpg')]
    deps = glob.glob(f'/content/build/llff/{s}/depth_maps/depth_*.png')
    summary[s] = dict(images=len(imgs), depth_maps=len(deps))
    assert len(imgs) == len(deps) > 0, (s, len(imgs), len(deps))
print(json.dumps(summary, indent=1))

# one archive to Drive, plus a small manifest so anyone can check they have the same data
!cd /content/build && tar -cf /content/build/llff_with_depth.tar llff/
md5 = subprocess.run(['md5sum', '/content/build/llff_with_depth.tar'], capture_output=True, text=True).stdout.split()[0]
!cp /content/build/llff_with_depth.tar {TAR}
json.dump(dict(md5=md5, scenes=summary, dpt='DPT_Hybrid (torch.hub intel-isl/MiDaS)', timm='0.6.13'),
          open(f'{ROOT}/data/llff_with_depth.manifest.json', 'w'), indent=1)
print('Uploaded', TAR, 'md5', md5)
!pip install -q -U timm   # back to the current timm for training

In [ ]:
# Cell 6: copy data to the fast local disk.
!mkdir -p /content/data && tar -xf {ROOT}/data/llff_with_depth.tar -C /content/data
!ls /content/data/llff

## Cell S: smoke test and timing (today's gate)

`ITERS = 500` checks that train, render and metrics all work. `ITERS = 6000` is a full run with the flags from
`scripts/run_llff.sh` and gives *m*, the minutes per run that decides 3 seeds vs 2. Post *m* to Aayush.

In [ ]:
# Cell S: smoke test / timing run on fern
import time, json, os
ITERS = 500
OUT = f'/content/smoke/fern_{ITERS}'
!rm -rf {OUT}
FLAGS = ('-r 8 --eval --n_sparse 3 --rand_pcd --lambda_dssim 0.2 --densify_grad_threshold 0.0013 '
         '--prune_threshold 0.01 --densify_until_iter 6000 --percent_dense 0.01 --position_lr_init 0.016 '
         '--position_lr_final 0.00016 --position_lr_max_steps 5500 --position_lr_start 500 '
         '--split_opacity_thresh 0.1 --error_tolerance 0.00025 --scaling_lr 0.003 '
         '--shape_pena 0.002 --opa_pena 0.001 --near 10')
t0 = time.time()
!python train_llff.py -s /content/data/llff/fern --model_path {OUT} --iterations {ITERS} {FLAGS}
t1 = time.time()
!python render.py -s /content/data/llff/fern --model_path {OUT} -r 8 --near 10
!python metrics.py --model_paths {OUT}
t2 = time.time()
import torch
print(f'train {(t1-t0)/60:.1f} min, render+metrics {(t2-t1)/60:.1f} min, total m = {(t2-t0)/60:.1f} min')
print(open(f'{OUT}/results_eval.json').read())   # DNGaussian writes results_eval.json, not results.json

In [ ]:
# Cell 7a (Disha, Tuesday): reproduce DNGaussian with the authors' own script. Cell 3 must use TAG = 'upstream-baseline'.
# run_llff.sh also renders a spiral video; that is harmless but adds a few minutes per scene.
SCENES = ['fern','flower','fortress','horns','leaves','orchids','room','trex']
for s in SCENES:
    out = f'{ROOT}/outputs/repro_upstream/{s}'
    if os.path.exists(f'{out}/results_eval.json'):
        print('done', s); continue
    !bash scripts/run_llff.sh /content/data/llff/{s} {out} 0 2>&1 | tail -5

In [ ]:
# Cell 7a-table: per-scene reproduction table against the paper (19.12 PSNR, 0.591 SSIM, 0.294 LPIPS)
import pandas as pd, json, os
rows = []
for s in SCENES:
    f = f'{ROOT}/outputs/repro_upstream/{s}/results_eval.json'
    if os.path.exists(f):
        m = next(iter(json.load(open(f)).values()))
        rows.append(dict(scene=s, psnr=m['PSNR'], ssim_gs=m['SSIM'], ssim_sk=m['SSIM_sk'], lpips=m['LPIPS']))
t = pd.DataFrame(rows).set_index('scene')
t.loc['mean'] = t.mean()
print(t.round(3)); print('paper: PSNR 19.12, SSIM 0.591, LPIPS 0.294')
t.to_csv(f'{ROOT}/outputs/repro_upstream/repro_table.csv')

In [ ]:
# Cell 7b (both, from Wed 14 Oct 1 PM): your shard of the main matrix. Cell 3 must use TAG = 'v1.0-exp'.
SHARD = '1/2'   # Aayush: 0/2, Disha: 1/2
os.makedirs(f'{ROOT}/logs', exist_ok=True)
LOG = f"{ROOT}/logs/colab_shard_{SHARD.replace('/', 'of')}.log"
!python experiments/run_matrix.py --matrix experiments/matrix.yaml --data_root /content/data/llff --out_root {ROOT}/outputs --shard {SHARD} --seeds 0 2>&1 | tee -a {LOG}